# Experimentos de Triagem PLN — CareSync

Notebook para testar o parser e o motor de regras com laudos de exemplo. Execute a partir da raiz do repositório (ou ajuste o `sys.path`).

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from src.parser import LabParser
from src.rules import MotorDeRegras

In [ ]:
laudos = {
    'normal': 'HEMOGRAMA\nHemoglobina: 13,5 g/dL (VR: 12,0 a 15,5)\nPlaquetas: 250.000 /mm³ (VR: 150.000 a 450.000)',
    'atencao': 'PERFIL LIPÍDICO\nColesterol total: 232 mg/dL (VR: < 190)\nGlicose: 118 mg/dL',
    'critico': 'ELETRÓLITOS\nPotássio: 6,6 mEq/L\nSódio: 138 mEq/L',
}

linhas = []
for nome, texto in laudos.items():
    dados = LabParser.parse_texto(texto)
    tipo = LabParser.detectar_tipo_exame(texto, [d['marcador'].id for d in dados])
    resultado = MotorDeRegras.avaliar_triagem_completa(dados, tipo)
    for item in resultado.exames_extraidos:
        linhas.append({'laudo': nome, 'tipo': resultado.tipo_exame, 'geral': resultado.classificacao_geral.value, **item.model_dump(mode='json')})

pd.DataFrame(linhas)